Anza Malik FA21-BCS-037

Pakistani Seasonal Color Analysis Random Forest Training

---



Step 1: Load the Dataset

In [ ]:
import pandas as pd

# Load the dataset
file_path = '/content/data.xlsx'  # Update this path if needed
df = pd.read_excel(file_path, sheet_name='Sheet1')

# Display the first few rows
print(df.head())

            Eye Color       Eyebrow Color             Skin Color  \
0  rgba(48,38,42,255)  rgba(14,16,28,255)  rgba(178,161,154,255)   
1  rgba(94,49,40,255)    rgba(97,67,63,1)  rgba(255,205,171,255)   
2  rgba(29,19,17,255)  rgba(50,38,26,255)   rgba(218,157,80,255)   
3   rgba(9,23,30,255)   rgba(9,23,30,255)  rgba(205,175,150,255)   
4  rgba(98,57,61,255)  rgba(36,24,51,255)  rgba(240,195,181,255)   

               Lip color    Season  
0  rgba(180,110,105,255)  "Autumn"  
1  rgba(220,115,105,255)  "Autumn"  
2   rgba(190,118,78,255)  "Autumn"  
3  rgba(180,110,105,255)  "Autumn"  
4  rgba(211,142,156,255)  "Autumn"  


Step 2: Clean the Dataset

In [ ]:
import re

# Function to clean RGBA values
def clean_rgba(rgba):
    rgba_values = re.sub(r'[^0-9,]', '', rgba)  # Remove non-numeric characters
    r, g, b, a = map(int, rgba_values.split(','))  # Extract R, G, B, A values
    return r, g, b  # Drop the alpha channel (A)

# Apply the function to each RGBA column
for col in ['Eye Color', 'Eyebrow Color', 'Skin Color', 'Lip color']:
    df[[f'{col}_R', f'{col}_G', f'{col}_B']] = df[col].apply(clean_rgba).apply(pd.Series)

# Remove quotes from the Season column
df['Season'] = df['Season'].str.strip('"')

# Drop the original RGBA columns
df = df.drop(columns=['Eye Color', 'Eyebrow Color', 'Skin Color', 'Lip color'])

# Verify the cleaned dataset
print(df.head())

   Season  Eye Color_R  Eye Color_G  Eye Color_B  Eyebrow Color_R  \
0  Autumn           48           38           42               14   
1  Autumn           94           49           40               97   
2  Autumn           29           19           17               50   
3  Autumn            9           23           30                9   
4  Autumn           98           57           61               36   

   Eyebrow Color_G  Eyebrow Color_B  Skin Color_R  Skin Color_G  Skin Color_B  \
0               16               28           178           161           154   
1               67               63           255           205           171   
2               38               26           218           157            80   
3               23               30           205           175           150   
4               24               51           240           195           181   

   Lip color_R  Lip color_G  Lip color_B  
0          180          110          105  
1          2

Step 3: Data Augmentation

In [ ]:
import numpy as np

# Function to augment data
def augment_data(df, num_augmentations=1):
    augmented_data = []
    for _, row in df.iterrows():
        for _ in range(num_augmentations):
            new_row = row.copy()
            for col in ['Eye Color_R', 'Eye Color_G', 'Eye Color_B',
                        'Eyebrow Color_R', 'Eyebrow Color_G', 'Eyebrow Color_B',
                        'Skin Color_R', 'Skin Color_G', 'Skin Color_B',
                        'Lip color_R', 'Lip color_G', 'Lip color_B']:
                # Add small random noise to R, G, B values
                new_row[col] = min(max(new_row[col] + np.random.randint(-5, 6), 0), 255)
            augmented_data.append(new_row)
    return pd.DataFrame(augmented_data)

# Augment the dataset (double the size)
augmented_df = augment_data(df, num_augmentations=1)
final_df = pd.concat([df, augmented_df], ignore_index=True)

# Verify the augmented dataset
print("Original dataset size:", len(df))
print("Augmented dataset size:", len(final_df))
print(final_df.head())

Original dataset size: 100
Augmented dataset size: 200
   Season  Eye Color_R  Eye Color_G  Eye Color_B  Eyebrow Color_R  \
0  Autumn           48           38           42               14   
1  Autumn           94           49           40               97   
2  Autumn           29           19           17               50   
3  Autumn            9           23           30                9   
4  Autumn           98           57           61               36   

   Eyebrow Color_G  Eyebrow Color_B  Skin Color_R  Skin Color_G  Skin Color_B  \
0               16               28           178           161           154   
1               67               63           255           205           171   
2               38               26           218           157            80   
3               23               30           205           175           150   
4               24               51           240           195           181   

   Lip color_R  Lip color_G  Lip color_B  


Step 4: Balance the Dataset

In [ ]:
# Count the number of instances per class
class_counts = final_df['Season'].value_counts()
print("Class counts before balancing:\n", class_counts)

# Find the minimum number of instances across all classes
min_count = class_counts.min()

# Sample equal instances from each class
balanced_df = final_df.groupby('Season').apply(lambda x: x.sample(min_count, random_state=42)).reset_index(drop=True)

# Verify the balanced dataset
print("\nClass counts after balancing:\n", balanced_df['Season'].value_counts())

Class counts before balancing:
 Season
Autumn    50
Spring    50
Winter    50
Summer    50
Name: count, dtype: int64

Class counts after balancing:
 Season
Autumn    50
Spring    50
Summer    50
Winter    50
Name: count, dtype: int64


<ipython-input-29-64e0f85a0c01>:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = final_df.groupby('Season').apply(lambda x: x.sample(min_count, random_state=42)).reset_index(drop=True)


Step 5: Add Brightness as a Feature

In [ ]:
# Function to calculate brightness
def calculate_brightness(r, g, b):
    return (0.299 * r + 0.587 * g + 0.114 * b) / 255

# Add brightness for each color feature
for col in ['Eye Color', 'Eyebrow Color', 'Skin Color', 'Lip color']:
    balanced_df[f'{col}_Brightness'] = calculate_brightness(balanced_df[f'{col}_R'], balanced_df[f'{col}_G'], balanced_df[f'{col}_B'])

# Verify the updated dataset
print(balanced_df.head())

   Season  Eye Color_R  Eye Color_G  Eye Color_B  Eyebrow Color_R  \
0  Autumn           25           17           15               60   
1  Autumn           33           28           21               42   
2  Autumn            6            5            6               47   
3  Autumn           55           36           35               97   
4  Autumn           46           42           40               53   

   Eyebrow Color_G  Eyebrow Color_B  Skin Color_R  Skin Color_G  Skin Color_B  \
0               53               50           195           168           151   
1               38               36           240           187           164   
2                7                6           175           138           127   
3               66               60           151           107            89   
4               44               43           191           142           110   

   Lip color_R  Lip color_G  Lip color_B  Eye Color_Brightness  \
0          172          125     

Step 6: Split the Dataset

In [ ]:
from sklearn.model_selection import train_test_split

# Features (X) and target (y)
X = balanced_df.drop(columns=['Season'])
y = balanced_df['Season']

# Split the dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Verify the split
print("Training set class distribution:\n", y_train.value_counts())
print("Testing set class distribution:\n", y_test.value_counts())

Training set class distribution:
 Season
Spring    40
Winter    40
Autumn    40
Summer    40
Name: count, dtype: int64
Testing set class distribution:
 Season
Autumn    10
Summer    10
Spring    10
Winter    10
Name: count, dtype: int64


Step 7: Train and Evaluate Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate the model
y_pred_rf = rf_model.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.925
Random Forest Classification Report:
              precision    recall  f1-score   support

      Autumn       1.00      1.00      1.00        10
      Spring       1.00      0.90      0.95        10
      Summer       0.89      0.80      0.84        10
      Winter       0.83      1.00      0.91        10

    accuracy                           0.93        40
   macro avg       0.93      0.93      0.92        40
weighted avg       0.93      0.93      0.92        40



In [ ]:
import joblib

# Save the trained model to a file
model_filename = 'random_forest_model.pkl'
joblib.dump(rf_model, model_filename)

print(f"Model saved to {model_filename}")

Model saved to random_forest_model.pkl
